# Inspección del dataset creado `.parquet`


En este notebook se inspeccionará el dataset creado en formato `.parquet` para verificar que su estructura sea correcta según la metodología de creación del dataset.

In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / 'src'))

from record_linkage.config import PROCESSED_DIR

PERFIL = "B1"  # Cambiar a "B2" para inspeccionar ese perfil

PARQUET = PROCESSED_DIR / PERFIL.lower() / 'dataset.parquet'
df = pd.read_parquet(PARQUET)
print(f'Parquet cargado: {PARQUET}')
print(f'Shape: {df.shape}')

Parquet cargado: /home/uzi/Data/INER/processed/b1/dataset.parquet
Shape: (23706, 4)


## 1. Estructura del parquet

Estadísticas básicas: columnas, registros por fuente, distribución de entity_ids.

In [8]:
# Columnas y dtypes
print('Columnas:')
print(df.dtypes)
print()

# Registros por source_db
print('Registros por fuente:')
print(df['source_db'].value_counts().to_string())
print(f'\nTotal: {len(df):,}')

Columnas:
record_id    int64
source_db      str
text           str
entity_id    int64
dtype: object

Registros por fuente:
source_db
Trabajo Social    14796
Económico          4632
Comorbilidad       4278

Total: 23,706


In [9]:
# Distribución de entity_ids
counts = df['entity_id'].value_counts()
singles   = (counts == 1).sum()
en_2      = (counts == 2).sum()
en_3      = (counts == 3).sum()
en_4_plus = (counts >= 4).sum()
vinculables = en_2 + en_3 + en_4_plus

print(f'Entidades únicas totales:           {counts.shape[0]:>7,}')
print(f'  Solo en 1 CSV (singletons):       {singles:>7,}')
print(f'  En exactamente 2 CSV:             {en_2:>7,}')
print(f'  En las 3 CSV:                     {en_3:>7,}')
print(f'  En 4+ registros (intra-dup):      {en_4_plus:>7,}')
print(f'  Vinculables (≥2 registros):       {vinculables:>7,}  ← esperado ~4,341')
print()
vinc_mask = df['entity_id'].isin(counts[counts > 1].index)
print(f'Registros en entidades vinculables: {vinc_mask.sum():,}')
print(f'Pares positivos potenciales:        {(counts[counts > 1] * (counts[counts > 1] - 1) // 2).sum():,}')


Entidades únicas totales:            16,222
  Solo en 1 CSV (singletons):        11,841
  En exactamente 2 CSV:               1,539
  En las 3 CSV:                       2,672
  En 4+ registros (intra-dup):          170
  Vinculables (≥2 registros):         4,381  ← esperado ~4,341

Registros en entidades vinculables: 11,865
Pares positivos potenciales:        10,996


In [10]:
# Pares positivos cross-database confirmados
# Entidades con registros en más de un source_db
cross_db = (
    df.groupby('entity_id')['source_db']
    .nunique()
    .reset_index()
    .rename(columns={'source_db': 'n_fuentes'})
)

vinculables = cross_db[cross_db['n_fuentes'] > 1]
print(f'Entidades con registros en ≥2 fuentes: {len(vinculables):,}  ← esperado 4,341')
print()
print('Distribución por número de fuentes:')
print(vinculables['n_fuentes'].value_counts().sort_index().to_string())


Entidades con registros en ≥2 fuentes: 4,341  ← esperado 4,341

Distribución por número de fuentes:
n_fuentes
2    1584
3    2757


In [5]:
# Pares positivos cross-database por cruce
fuentes = ['Comorbilidad', 'Económico', 'Trabajo Social']
cruces = [
    ('Comorbilidad',  'Económico'),
    ('Comorbilidad',  'Trabajo Social'),
    ('Económico',     'Trabajo Social'),
]

total_conf = 0
for a, b in cruces:
    ids_a = set(df[df['source_db'] == a]['entity_id'])
    ids_b = set(df[df['source_db'] == b]['entity_id'])
    conf  = len(ids_a & ids_b)
    total_conf += conf
    print(f'{a} ↔ {b}: {conf:,}')

print(f'\nTotal confirmados: {total_conf:,}  ← esperado 9,855')


Comorbilidad ↔ Económico: 2,939
Comorbilidad ↔ Trabajo Social: 3,187
Económico ↔ Trabajo Social: 3,729

Total confirmados: 9,855  ← esperado 9,855


## 2. Inspección de serialización — Perfil Zero-Shot (sin tokens especiales)

Texto tal cual lo recibe el modelo: formato `columna: valor` sin tokens de bloque. 2 registros por fuente.

In [3]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent / 'src'))
from record_linkage.config import PROCESSED_DIR
PERFIL = "ZERO_SHOT"  # Cambiar para inspeccionar otro perfil

PARQUET = PROCESSED_DIR / PERFIL.lower() / 'dataset.parquet'
df_zs = pd.read_parquet(PARQUET)
print(f'Parquet cargado: {PARQUET}')
print(f'Shape: {df_zs.shape}')

Parquet cargado: /home/uzi/Data/INER/processed/zero_shot/dataset.parquet
Shape: (23706, 4)


In [4]:
muestra = df_zs[df_zs['source_db'] == 'Comorbilidad'].head(2)
for _, row in muestra.iterrows():
    print(f'[record_id={row["record_id"]}  entity_id={row["entity_id"]}]')
    print(row['text'])
    print()

[record_id=0  entity_id=0]
expediente: 237124 nombre: ANGEL OCTAVIO    GUERRERO    SANCHEZ fechaing: 2020-02-28 fechaegr: 2020-03-02 diagnosticoprincipal: COVID 19 cie101: U07.1 diagnostico2: NODULO PULMONAR cie102: J98.4 dx2: J98.4 obesidad: 0 obesidad1: 0 cardiopatia: 0 comorbi: Otras diabetes: 0 nefropatia: 0 eaperge: 0 tephap: 0 comorbicv: 0

[record_id=1  entity_id=1]
expediente: 237346 nombre: FELIPE           GONZALEZ    GUTIERREZ fechaing: 2020-03-14 fechaegr: 2020-03-18 diagnosticoprincipal: SARS COV2 cie101: U07.1 obesidad: 0 obesidad1: 0 cardiopatia: 0 comorbi: Ninguna diabetes: 0 nefropatia: 0 eaperge: 0 tephap: 0 comorbicv: 0



In [5]:
muestra = df_zs[df_zs['source_db'] == 'Económico'].head(2)
for _, row in muestra.iterrows():
    print(f'[record_id={row["record_id"]}  entity_id={row["entity_id"]}]')
    print(row['text'])
    print()

[record_id=4278  entity_id=4108]
EXP: 248230 NOMBRE_DEL_PACIENTE: RUIZ MARTINEZ HILARIO WILFRIDO SEXO: M EDAD: 76 GRUPO_EDAD: 75-79 RESULTADO: SARS-COV2 ETIQUETAS_COVID: COVID+RVCOVID MOTIVO_DE_EGRESO: ALTA POR MEJORIA FECHA_INGRESO_INER: 2023-01-07 FECHA_DE_ALTA_MEJORIA: 2023-01-18 DIAS_ESTANCIA: 11 GASTO_TOTAL: 201731.14 GASTO_DIARIO: 18339.194545454546 DERECHOHABIENTE_Y/O_BENEFICIARIO: NINGUNO VULNERABILIDAD_SOCIOECONOMICA: 0 ESTADO_RESIDENCIA: CIUDAD DE MEXICO CLAVE_GEOESTADISTICA_ESTATAL: 9 MUNICIPIO_RESIDENCIA: TLALPAN CLAVE_GEOESTADISTICA_MUNICIPAL: 12

[record_id=4279  entity_id=4109]
EXP: 248237 NOMBRE_DEL_PACIENTE: MERCADO MAGDALENO LAURA ANGELICA SEXO: F EDAD: 54 GRUPO_EDAD: 50-54 RESULTADO: SARS-COV2 ETIQUETAS_COVID: COVID+NOVACOVID MOTIVO_DE_EGRESO: ALTA POR MEJORIA FECHA_INGRESO_INER: 2023-01-01 FECHA_DE_ALTA_MEJORIA: 2023-01-06 DIAS_ESTANCIA: 5 GASTO_TOTAL: 93812.67 GASTO_DIARIO: 18762.534000000003 DERECHOHABIENTE_Y/O_BENEFICIARIO: NINGUNO VULNERABILIDAD_SOCIOECONOMICA: 

In [7]:
muestra = df_zs[df_zs['source_db'] == 'Trabajo Social'].head(2)
for _, row in muestra.iterrows():
    print(f'[record_id={row["record_id"]}  entity_id={row["entity_id"]}]')
    print(row['text'])
    print()

[record_id=8910  entity_id=5586]
AÑO: 0 FILA: 0 EXPEDIENTE: 236215 NO. HISTORIA: IAN600536 FECHA DE ELABORACIÓN: 2020-01-01 10:51:00 APELLIDO PATERNO: CORONA APELLIDO MATERNO: LOPEZ NOMBRE: SHEILA YADIRA EDAD: 45 años 2 meses 22 días FECHA DE NACIMIENTO: 1974-10-11 GENERO: Femenino DIAGNOSTICO: Estado asmático ESCOLARIDAD: NMS INCOMPLETO OCUPACIÓN: Hogar DERECHOHABIENTE Y/O BENEFICIARIO: SEG.POP. DELEGACIÓN O MUNICIPIO PERMANENTE: Tlalnepantla de Baz ESTADO / PAIS PERMANENTE: México,Mexico TOTAL DE PUNTOS: 27 NIVEL SOCIOECONÓMICO: 2

[record_id=8911  entity_id=5587]
AÑO: 0 FILA: 1 EXPEDIENTE: 236216 NO. HISTORIA: IAN600533 FECHA DE ELABORACIÓN: 2020-01-01 07:05:00 APELLIDO PATERNO: ZAMBRANO APELLIDO MATERNO: RIVAS NOMBRE: JOSE LUIS EDAD: 70 años 3 meses 20 días FECHA DE NACIMIENTO: 1949-09-13 GENERO: Masculino DIAGNOSTICO: Anormalidades de la respiración ESCOLARIDAD: PRIMARIA COMPLETA OCUPACIÓN: DESEMPLEADO DERECHOHABIENTE Y/O BENEFICIARIO: NINGUNO DELEGACIÓN O MUNICIPIO PERMANENTE: Iz

## 3. Inspección de serialización — Perfil B1 (con tokens especiales)

Se muestran 2 registros por CSV para verificar bloques semánticos, tokens `[BLK_*]`, `[COL]`, `[VAL]` y formato de valores. Los bloques sin datos deben estar ausentes.


In [1]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent / 'src'))
from record_linkage.config import PROCESSED_DIR

PERFIL = "B1"  # Cambiar para inspeccionar otro perfil

PARQUET = PROCESSED_DIR / PERFIL.lower() / 'dataset.parquet'
df = pd.read_parquet(PARQUET)
print(f'Parquet cargado: {PARQUET}')
print(f'Shape: {df.shape}')

Parquet cargado: /home/uzi/Data/INER/processed/b1/dataset.parquet
Shape: (23706, 4)


In [2]:
source = 'Comorbilidad'
print(f'  {source}')
print(f'{"=" * 70}')
muestra = df[df['source_db'] == source].head(2)
text1 = muestra.iloc[0]['text']
text2 = muestra.iloc[1]['text']

print('--- Registro 1 ---')
print(text1)
print('\n--- Registro 2 ---')
print(text2)

  Comorbilidad
--- Registro 1 ---
[BLK_ID] [COL] nombre [VAL] ANGEL OCTAVIO    GUERRERO    SANCHEZ [BLK_CLIN] [COL] diagnosticoprincipal [VAL] COVID 19 [COL] cie101 [VAL] U07.1 [COL] diagnostico2 [VAL] NODULO PULMONAR [COL] cie102 [VAL] J98.4 [COL] dx2 [VAL] J98.4 [COL] diagnostico3 [VAL] NULL [COL] cie103 [VAL] NULL [COL] dx3 [VAL] NULL [COL] diagnostico4 [VAL] NULL [COL] cie104 [VAL] NULL [COL] dx4 [VAL] NULL [COL] comorbi [VAL] Otras [COL] comorbicv [VAL] 0 [COL] obesidad [VAL] 0 [COL] obesidad1 [VAL] 0 [COL] cardiopatia [VAL] 0 [COL] diabetes [VAL] 0 [COL] nefropatia [VAL] 0 [COL] eaperge [VAL] 0 [COL] tephap [VAL] 0 [BLK_ADMIN] [COL] expediente [VAL] 237124 [COL] fechaing [VAL] 2020-02-28 [COL] fechaegr [VAL] 2020-03-02

--- Registro 2 ---
[BLK_ID] [COL] nombre [VAL] FELIPE           GONZALEZ    GUTIERREZ [BLK_CLIN] [COL] diagnosticoprincipal [VAL] SARS COV2 [COL] cie101 [VAL] U07.1 [COL] diagnostico2 [VAL] NULL [COL] cie102 [VAL] NULL [COL] dx2 [VAL] NULL [COL] diagnostico3 [VAL]

In [4]:
source = 'Comorbilidad'
print(f'  {source}')
print(f'{"=" * 70}')
muestra = df[df['source_db'] == source].head(2)
for _, row in muestra.iterrows():
    print(f'\n[record_id={row["record_id"]}  entity_id={row["entity_id"]}]')
    text = row['text']
    tokens_bloque = ['[BLK_ID]', '[BLK_CLIN]', '[BLK_ADMIN]', '[BLK_GEO]', '[BLK_SOCIO]']
    posiciones = [(b, text.index(b)) for b in tokens_bloque if b in text]
    posiciones.sort(key=lambda x: x[1])
    for i, (bloque, start) in enumerate(posiciones):
        end = posiciones[i + 1][1] if i + 1 < len(posiciones) else len(text)
        print(f'  {text[start:end].strip()}')
print()


  Comorbilidad

[record_id=0  entity_id=0]
  [BLK_ID] [COL] nombre [VAL] ANGEL OCTAVIO    GUERRERO    SANCHEZ
  [BLK_CLIN] [COL] diagnosticoprincipal [VAL] COVID 19 [COL] cie101 [VAL] U07.1 [COL] diagnostico2 [VAL] NODULO PULMONAR [COL] cie102 [VAL] J98.4 [COL] dx2 [VAL] J98.4 [COL] diagnostico3 [VAL] NULL [COL] cie103 [VAL] NULL [COL] dx3 [VAL] NULL [COL] diagnostico4 [VAL] NULL [COL] cie104 [VAL] NULL [COL] dx4 [VAL] NULL [COL] comorbi [VAL] Otras [COL] comorbicv [VAL] 0 [COL] obesidad [VAL] 0 [COL] obesidad1 [VAL] 0 [COL] cardiopatia [VAL] 0 [COL] diabetes [VAL] 0 [COL] nefropatia [VAL] 0 [COL] eaperge [VAL] 0 [COL] tephap [VAL] 0
  [BLK_ADMIN] [COL] expediente [VAL] 237124 [COL] fechaing [VAL] 2020-02-28 [COL] fechaegr [VAL] 2020-03-02

[record_id=1  entity_id=1]
  [BLK_ID] [COL] nombre [VAL] FELIPE           GONZALEZ    GUTIERREZ
  [BLK_CLIN] [COL] diagnosticoprincipal [VAL] SARS COV2 [COL] cie101 [VAL] U07.1 [COL] diagnostico2 [VAL] NULL [COL] cie102 [VAL] NULL [COL] dx2 [VAL] NU

In [5]:
source = 'Económico'
print(f'  {source}')
print(f'{"=" * 70}')
muestra = df[df['source_db'] == source].head(2)
for _, row in muestra.iterrows():
    print(f'\n[record_id={row["record_id"]}  entity_id={row["entity_id"]}]')
    text = row['text']
    tokens_bloque = ['[BLK_ID]', '[BLK_CLIN]', '[BLK_ADMIN]', '[BLK_GEO]', '[BLK_SOCIO]']
    posiciones = [(b, text.index(b)) for b in tokens_bloque if b in text]
    posiciones.sort(key=lambda x: x[1])
    for i, (bloque, start) in enumerate(posiciones):
        end = posiciones[i + 1][1] if i + 1 < len(posiciones) else len(text)
        print(f'  {text[start:end].strip()}')
print()

  Económico

[record_id=4278  entity_id=4108]
  [BLK_ID] [COL] NOMBRE_DEL_PACIENTE [VAL] RUIZ MARTINEZ HILARIO WILFRIDO [COL] SEXO [VAL] M [COL] EDAD [VAL] 76 [COL] GRUPO_EDAD [VAL] 75-79
  [BLK_CLIN] [COL] RESULTADO [VAL] SARS-COV2 [COL] ETIQUETAS_COVID [VAL] COVID+RVCOVID [COL] MOTIVO_DE_EGRESO [VAL] ALTA POR MEJORIA
  [BLK_ADMIN] [COL] EXP [VAL] 248230 [COL] DIAS_ESTANCIA [VAL] 11 [COL] FECHA_INGRESO_INER [VAL] 2023-01-07 [COL] FECHA_DE_ALTA_MEJORIA [VAL] 2023-01-18 [COL] TOTAL_DE_INGRESOS [VAL] NULL [COL] TOTAL_DE_EGRESOS [VAL] NULL [COL] GASTO_TOTAL [VAL] 201731.14 [COL] GASTO_DIARIO [VAL] 18339.194545454546
  [BLK_GEO] [COL] ESTADO_RESIDENCIA [VAL] CIUDAD DE MEXICO [COL] CLAVE_GEOESTADISTICA_ESTATAL [VAL] 9 [COL] MUNICIPIO_RESIDENCIA [VAL] TLALPAN [COL] CLAVE_GEOESTADISTICA_MUNICIPAL [VAL] 12
  [BLK_SOCIO] [COL] ESCOLARIDAD [VAL] NULL [COL] OCUPACION [VAL] NULL [COL] VULNERABILIDAD_SOCIOECONOMICA [VAL] 0 [COL] NIVEL_SOCIOECONOMICO [VAL] NULL [COL] DERECHOHABIENTE_Y/O_BENEFICIARIO

In [6]:
source = 'Trabajo Social'
print(f'  {source}')
print(f'{"=" * 70}')
muestra = df[df['source_db'] == source].head(2)
for _, row in muestra.iterrows():
    print(f'\n[record_id={row["record_id"]}  entity_id={row["entity_id"]}]')
    text = row['text']
    tokens_bloque = ['[BLK_ID]', '[BLK_CLIN]', '[BLK_ADMIN]', '[BLK_GEO]', '[BLK_SOCIO]']
    posiciones = [(b, text.index(b)) for b in tokens_bloque if b in text]
    posiciones.sort(key=lambda x: x[1])
    for i, (bloque, start) in enumerate(posiciones):
        end = posiciones[i + 1][1] if i + 1 < len(posiciones) else len(text)
        print(f'  {text[start:end].strip()}')
print()

  Trabajo Social

[record_id=8910  entity_id=5586]
  [BLK_ID] [COL] APELLIDO PATERNO [VAL] CORONA [COL] APELLIDO MATERNO [VAL] LOPEZ [COL] NOMBRE [VAL] SHEILA YADIRA [COL] EDAD [VAL] 45 años 2 meses 22 días [COL] FECHA DE NACIMIENTO [VAL] 1974-10-11 [COL] GENERO [VAL] Femenino
  [BLK_CLIN] [COL] DIAGNOSTICO [VAL] Estado asmático
  [BLK_ADMIN] [COL] EXPEDIENTE [VAL] 236215 [COL] NO. HISTORIA [VAL] IAN600536 [COL] FECHA DE ELABORACIÓN [VAL] 2020-01-01 10:51:00 [COL] AÑO [VAL] 0 [COL] FILA [VAL] 0
  [BLK_GEO] [COL] DELEGACIÓN O MUNICIPIO PERMANENTE [VAL] Tlalnepantla de Baz [COL] ESTADO / PAIS PERMANENTE [VAL] México,Mexico
  [BLK_SOCIO] [COL] ESCOLARIDAD [VAL] NMS INCOMPLETO [COL] OCUPACIÓN [VAL] Hogar [COL] DERECHOHABIENTE Y/O BENEFICIARIO [VAL] SEG.POP. [COL] TOTAL DE PUNTOS [VAL] 27 [COL] NIVEL SOCIOECONÓMICO [VAL] 2

[record_id=8911  entity_id=5587]
  [BLK_ID] [COL] APELLIDO PATERNO [VAL] ZAMBRANO [COL] APELLIDO MATERNO [VAL] RIVAS [COL] NOMBRE [VAL] JOSE LUIS [COL] EDAD [VAL] 70 año

## 4. Distribución de longitud en tokens (por tokenizador)

Cuántos tokens genera cada tokenizador sobre los textos zero-shot. Referencia: truncación a 128 y 512 tokens.

In [2]:
from transformers import AutoTokenizer
from record_linkage.config import MODELS_DIR

tokenizers = {
    "BETO":                    AutoTokenizer.from_pretrained(str(MODELS_DIR / "pretrained" / "BETO")),
    "RoBERTa-biomedical":      AutoTokenizer.from_pretrained(str(MODELS_DIR / "pretrained" / "RoBERTa-biomedical")),
    "paraphrase-multilingual": AutoTokenizer.from_pretrained(str(MODELS_DIR / "pretrained" / "paraphrase-multilingual")),
}
print("Tokenizadores cargados.")

/home/uzi/micromamba/envs/tesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizadores cargados.


In [3]:
import numpy as np

for model_name, tokenizer in tokenizers.items():
    print(f"{model_name}")
    print(f"{'='*60}")
    lengths = df["text"].apply(lambda t: len(tokenizer.encode(t, add_special_tokens=True)))
    for source in ["Comorbilidad", "Económico", "Trabajo Social"]:
        l = lengths[df["source_db"] == source]
        print(f"\n  {source}  (n={len(l):,})")
        print(f"    min={l.min()}  media={l.mean():.1f}  max={l.max()}")


BETO

  Comorbilidad  (n=4,278)
    min=190  media=213.8  max=255

  Económico  (n=4,632)
    min=268  media=350.0  max=377

  Trabajo Social  (n=14,796)
    min=146  media=238.4  max=276
RoBERTa-biomedical

  Comorbilidad  (n=4,278)
    min=214  media=232.4  max=260

  Económico  (n=4,632)
    min=268  media=347.2  max=370

  Trabajo Social  (n=14,796)
    min=144  media=235.5  max=269
paraphrase-multilingual

  Comorbilidad  (n=4,278)
    min=163  media=185.8  max=223

  Económico  (n=4,632)
    min=222  media=294.6  max=311

  Trabajo Social  (n=14,796)
    min=127  media=208.5  max=243
